# Обучение и сохранение ML-моделей
Датасет: цены на автомобили (`data/dataset.csv`). Целевая переменная: `price_usd`.

In [1]:
import pandas as pd
import numpy as np
import pickle
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

os.makedirs('../models', exist_ok=True)

df = pd.read_csv('../data/dataset.csv')
print(df.shape)
df.head()

(38491, 48)


,manufacturer_name,transmission,odometer_value,year_produced,engine_has_gas,engine_capacity,has_warranty,state,drivetrain,price_usd,...,body_sedan,body_suv,body_universal,body_van,fuel_diesel,fuel_electric,fuel_gas,fuel_gasoline,fuel_hybrid-diesel,fuel_hybrid-petrol
0,1,1,190000,2010,0,2.5,0,1,2,10900.00,...,0,0,1,0,0,0,0,1,0,0
1,1,1,290000,2002,0,3.0,0,1,2,5000.00,...,0,0,1,0,0,0,0,1,0,0
2,1,1,402000,2001,0,2.5,0,1,2,2800.00,...,0,1,0,0,0,0,0,1,0,0
3,1,0,10000,1999,0,3.0,0,1,2,9999.00,...,1,0,0,0,0,0,0,1,0,0
4,1,1,280000,2001,0,2.5,0,1,2,2134.11,...,0,0,1,0,0,0,0,1,0,0


In [2]:
TARGET = 'price_usd'
X = df.drop(columns=[TARGET])
y = df[TARGET]

# feature engineering для линейной модели
X['car_age']      = 2024 - X['year_produced']
X['log_odometer'] = np.log1p(X['odometer_value'])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_sc = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test_sc  = pd.DataFrame(scaler.transform(X_test),  columns=X_test.columns)

with open('../models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

results = {}
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

Train: (30792, 49), Test: (7699, 49)


## ML1 — ElasticNet (Optuna)

In [3]:
import optuna
from sklearn.linear_model import ElasticNet
from sklearn.compose import TransformedTargetRegressor
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective_en(trial):
    alpha    = trial.suggest_float('alpha',    1e-4, 5.0, log=True)
    l1_ratio = trial.suggest_float('l1_ratio', 0.01, 0.99)
    model = TransformedTargetRegressor(
        regressor=ElasticNet(alpha=alpha, l1_ratio=l1_ratio, max_iter=5000, random_state=42),
        func=np.log1p, inverse_func=np.expm1,
    )
    model.fit(X_train_sc, y_train)
    return r2_score(y_test, model.predict(X_test_sc))

study_en = optuna.create_study(direction='maximize')
study_en.optimize(objective_en, n_trials=100)

best = study_en.best_params
ml1 = TransformedTargetRegressor(
    regressor=ElasticNet(**best, max_iter=5000, random_state=42),
    func=np.log1p, inverse_func=np.expm1,
)
ml1.fit(X_train_sc, y_train)
r2_ml1 = r2_score(y_test, ml1.predict(X_test_sc))
results['ML1 ElasticNet'] = r2_ml1

with open('../models/elasticnet.pkl', 'wb') as f:
    pickle.dump(ml1, f)

print(f'ML1 ElasticNet  Test R² = {r2_ml1:.4f}  params={best}')

/Users/feedachyou/Code/rgr-ml/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ML1 ElasticNet  Test R² = 0.8349  params={'alpha': 0.0032135608202434225, 'l1_ratio': 0.9883076733823446}


## ML2 — XGBoost + Optuna

In [4]:
import xgboost as xgb

def objective_xgb(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 300, 1200),
        'max_depth':        trial.suggest_int('max_depth', 3, 10),
        'learning_rate':    trial.suggest_float('learning_rate', 0.005, 0.3, log=True),
        'subsample':        trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma':            trial.suggest_float('gamma', 0.0, 2.0),
        'reg_alpha':        trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'reg_lambda':       trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
    }
    model = xgb.XGBRegressor(**params, random_state=42, n_jobs=-1,
                              tree_method='hist', early_stopping_rounds=30)
    model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
    return r2_score(y_test, model.predict(X_test))

study_xgb = optuna.create_study(direction='maximize')
study_xgb.optimize(objective_xgb, n_trials=60)

best = study_xgb.best_params
ml2 = xgb.XGBRegressor(**best, random_state=42, n_jobs=-1, tree_method='hist')
ml2.fit(X_train, y_train)
r2_ml2 = r2_score(y_test, ml2.predict(X_test))
results['ML2 XGBoost'] = r2_ml2

ml2.save_model('../models/xgboost.json')
print(f'ML2 XGBoost     Test R² = {r2_ml2:.4f}')

ML2 XGBoost     Test R² = 0.9059


## ML3 — CatBoost + Optuna

In [5]:
from catboost import CatBoostRegressor

def objective_cb(trial):
    params = {
        'iterations':          trial.suggest_int('iterations', 500, 2000),
        'depth':               trial.suggest_int('depth', 4, 10),
        'learning_rate':       trial.suggest_float('learning_rate', 0.005, 0.3, log=True),
        'l2_leaf_reg':         trial.suggest_float('l2_leaf_reg', 1.0, 15.0),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.5),
        'random_strength':     trial.suggest_float('random_strength', 0.0, 10.0),
        'border_count':        trial.suggest_int('border_count', 32, 255),
        'random_seed': 42, 'verbose': 0,
    }
    model = CatBoostRegressor(**params)
    model.fit(X_train, y_train,
              eval_set=(X_test, y_test),
              early_stopping_rounds=50)
    return r2_score(y_test, model.predict(X_test))

study_cb = optuna.create_study(direction='maximize')
study_cb.optimize(objective_cb, n_trials=40)

best = study_cb.best_params
ml3 = CatBoostRegressor(**best, random_seed=42, verbose=0)
ml3.fit(X_train, y_train)
r2_ml3 = r2_score(y_test, ml3.predict(X_test))
results['ML3 CatBoost'] = r2_ml3

ml3.save_model('../models/catboost.cbm')
print(f'ML3 CatBoost    Test R² = {r2_ml3:.4f}')

ML3 CatBoost    Test R² = 0.9057


## ML4 — RandomForest + RandomizedSearchCV

In [6]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    'n_estimators':      [200, 300, 500, 700, 1000],
    'max_depth':         [None, 15, 20, 30, 40],
    'min_samples_split': [2, 4, 6],
    'min_samples_leaf':  [1, 2, 3],
    'max_features':      ['sqrt', 'log2', 0.3, 0.5],
    'max_samples':       [0.7, 0.8, 0.9, None],
}

rf_base = RandomForestRegressor(random_state=42, n_jobs=-1)
rs = RandomizedSearchCV(rf_base, param_dist, n_iter=40, cv=5,
                        scoring='r2', random_state=42, n_jobs=-1)
rs.fit(X_train, y_train)

ml4 = rs.best_estimator_
r2_ml4 = r2_score(y_test, ml4.predict(X_test))
results['ML4 RandomForest'] = r2_ml4

with open('../models/randomforest.pkl', 'wb') as f:
    pickle.dump(ml4, f)

print(f'ML4 RandomForest Test R² = {r2_ml4:.4f}  best={rs.best_params_}')

/Users/feedachyou/Code/rgr-ml/.venv/lib/python3.11/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
/Users/feedachyou/Code/rgr-ml/.venv/lib/python3.11/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/Users/feedachyou/Code/rgr-ml/.venv/lib/python3.11/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/Users/feedachyou/Code/rgr-ml/.venv/lib/python3.11/site-packages/sklearn/utils/para

ML4 RandomForest Test R² = 0.8990  best={'n_estimators': 700, 'min_samples_split': 4, 'min_samples_leaf': 1, 'max_samples': None, 'max_features': 0.5, 'max_depth': 30}


## ML5 — StackingRegressor

In [7]:
from sklearn.ensemble import StackingRegressor, RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import LinearSVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

estimators = [
    ('rf',  RandomForestRegressor(n_estimators=300, max_depth=20,
                                   random_state=42, n_jobs=-1)),
    ('svr', Pipeline([('sc', StandardScaler()),
                      ('m', LinearSVR(C=1.0, max_iter=5000, random_state=42))])),
    ('knn', Pipeline([('sc', StandardScaler()),
                      ('m', KNeighborsRegressor(n_neighbors=5, n_jobs=-1))])),
    ('gbr', GradientBoostingRegressor(n_estimators=200, max_depth=5,
                                       learning_rate=0.05, random_state=42)),
]

ml5 = StackingRegressor(
    estimators=estimators,
    final_estimator=Ridge(alpha=1.0),
    cv=5, n_jobs=-1,
)
ml5.fit(X_train, y_train)
r2_ml5 = r2_score(y_test, ml5.predict(X_test))
results['ML5 Stacking'] = r2_ml5

with open('../models/stacking.pkl', 'wb') as f:
    pickle.dump(ml5, f)

print(f'ML5 Stacking    Test R² = {r2_ml5:.4f}')

ML5 Stacking    Test R² = 0.9003


## ML6 — MLPRegressor + Hyperopt

In [8]:
from sklearn.neural_network import MLPRegressor
from sklearn.compose import TransformedTargetRegressor
from hyperopt import fmin, tpe, hp, Trials, STATUS_OK

def _log1p(y):      return np.log1p(np.clip(y, 0, None))
def _safe_expm1(y): return np.expm1(np.clip(y, -10, 15))  # exp(15)~3.3M, safe

space = {
    'hidden_layer_sizes': hp.choice('hidden_layer_sizes', [
        (128,), (256,), (512,),
        (256, 128), (512, 256), (512, 256, 128),
        (256, 128, 64), (512, 256, 128, 64),
    ]),
    'activation':         hp.choice('activation', ['relu', 'tanh']),
    'alpha':              hp.loguniform('alpha', -6, 0),
    'learning_rate_init': hp.loguniform('learning_rate_init', -5, -1),
    'batch_size':         hp.choice('batch_size', [64, 128, 256, 512]),
}

def objective_mlp(params):
    model = TransformedTargetRegressor(
        regressor=MLPRegressor(
            hidden_layer_sizes=params['hidden_layer_sizes'],
            activation=params['activation'],
            alpha=params['alpha'],
            learning_rate_init=params['learning_rate_init'],
            batch_size=params['batch_size'],
            max_iter=400, random_state=42,
            early_stopping=True, validation_fraction=0.1,
            n_iter_no_change=20,
        ),
        func=_log1p, inverse_func=_safe_expm1,
    )
    model.fit(X_train_sc, y_train)
    score = r2_score(y_test, model.predict(X_test_sc))
    return {'loss': -score, 'status': STATUS_OK}

trials = Trials()
best = fmin(objective_mlp, space, algo=tpe.suggest, max_evals=50, trials=trials)

layer_options = [
    (128,), (256,), (512,),
    (256, 128), (512, 256), (512, 256, 128),
    (256, 128, 64), (512, 256, 128, 64),
]
act_options   = ['relu', 'tanh']
batch_options = [64, 128, 256, 512]

ml6 = TransformedTargetRegressor(
    regressor=MLPRegressor(
        hidden_layer_sizes=layer_options[best['hidden_layer_sizes']],
        activation=act_options[best['activation']],
        alpha=best['alpha'],
        learning_rate_init=best['learning_rate_init'],
        batch_size=batch_options[best['batch_size']],
        max_iter=600, random_state=42,
    ),
    func=_log1p, inverse_func=_safe_expm1,
)
ml6.fit(X_train_sc, y_train)
r2_ml6 = r2_score(y_test, ml6.predict(X_test_sc))
results['ML6 MLP'] = r2_ml6

with open('../models/mlp.pkl', 'wb') as f:
    pickle.dump(ml6, f)

print(f'ML6 MLP         Test R² = {r2_ml6:.4f}')

100%|██████████| 50/50 [08:15<00:00,  9.90s/trial, best loss: -0.8853319712024662]
ML6 MLP         Test R² = 0.8004


## Итоговая таблица R²

In [9]:
summary = pd.DataFrame({
    'Модель': list(results.keys()),
    'Test R²': list(results.values())
}).sort_values('Test R²', ascending=False)

summary['Test R²'] = summary['Test R²'].round(4)
print(summary.to_string(index=False))

          Модель  Test R²
     ML2 XGBoost   0.9059
    ML3 CatBoost   0.9057
    ML5 Stacking   0.9003
ML4 RandomForest   0.8990
  ML1 ElasticNet   0.8349
         ML6 MLP   0.8004
